# 1 · Your first quest — the Poisson problem

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=01-poisson.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/01-poisson.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>


Your masters are not cruel: for your **first quest** they smoothed the path and lend
a hand. They want you to see, *once and whole*, the **five deeds every adventurer
must master** to bend the Beast to their will:

1. cast a model into **variational (weak) form**,
2. conjure a **geometry and a mesh**,
3. wield **evaluable functions on the mesh** (*CoefficientFunctions*),
4. forge a **discrete problem** with the finite element method (**FEM**), and
5. **solve** the resulting system.

Each later unit drills one of these deeds. Today we walk all five on the gentlest
problem there is — the **Poisson equation** on the unit square $\Omega=(0,1)^2$,
$$ -\Delta u = f \ \text{ in } \Omega, \qquad u = 0 \ \text{ on } \partial\Omega. $$

### Deed 1 — the variational form

We never solve the strong equation directly. Multiplying by a test function $v$
and integrating by parts gives the **weak form**: find $u\in H_0^1(\Omega)$ with
$$ \underbrace{\int_\Omega \nabla u\cdot\nabla v\,dx}_{a(u,v)}
   = \underbrace{\int_\Omega f\,v\,dx}_{f(v)} \qquad\forall\,v\in H_0^1(\Omega). $$
This is the language NGSolve speaks — every problem in this course is a weak form.

### Deed 2 — geometry & mesh

`unit_square` is a built-in geometry; Netgen triangulates it at size `maxh`.
(Next unit conjures geometries of your own.)

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
Draw(mesh)

### Deed 3 — functions on the mesh, in a finite element space

We approximate $u$ in a **finite element space**: $H^1$-conforming, polynomial
order 3. The `dirichlet` flag marks the boundary edges where $u$ is fixed
(essential boundary condition).

In [ ]:
fes = H1(mesh, order=3, dirichlet="left|right|bottom|top")
print("degrees of freedom:", fes.ndof)

### Deed 4 — the discrete problem (FEM)

Trial and test functions turn $a(\cdot,\cdot)$ and $f(\cdot)$ into a **matrix** and
a **vector**. We pick $f=32\,(y(1-y)+x(1-x))$, so the exact answer is the bump
$u=16\,x(1-x)\,y(1-y)$ — handy for checking ourselves later.

In [ ]:
u, v = fes.TnT()                                     # trial & test functions
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
f = LinearForm(32 * (y * (1 - y) + x * (1 - x)) * v * dx).Assemble()

### Deed 5 — solve

Invert the system matrix on the **free** degrees of freedom (the non-Dirichlet
ones). `sparsecholesky` is the symmetric direct solver — also available in
JupyterLite/WebAssembly. (Unit 5 returns to *how* this solve really works.)

In [ ]:
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
Draw(gfu, mesh, "u")

## Did we do it right?

Because we *chose* the exact solution, we can measure the error in the $L^2$ norm —
our first taste of **deed 6 that nobody warns you about: always check your work.**

In [ ]:
exact = 16 * x * (1 - x) * y * (1 - y)
print(f"L2 error: {sqrt(Integrate((gfu - exact)**2, mesh)):.3e}")

Five deeds, one solved PDE. The Beast is unimpressed — but the masters whisper:
*before you can command it, you must truly command the **ground** — geometry and
mesh.* That is the next unit.